<a href="https://colab.research.google.com/github/Icarusleo/Medical-Ai-Agent/blob/main/Scripts/Training/finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Llama 3 ve QLoRA için gerekli temel kütüphaneler
# -q (quiet) parametresi gereksiz çıktıları gizler
!pip install -q -U torch torchvision torchaudio
!pip install -q -U transformers>=4.40.0  # Llama 3 desteği için kritik
!pip install -q -U datasets               # Veri setlerini yönetmek için
!pip install -q -U accelerate             # Modeli GPU'ya dağıtmak için
!pip install -q -U bitsandbytes           # 4-bit quantization için (Olmazsa olmaz)
!pip install -q -U peft                   # LoRA adaptörleri için
!pip install -q -U trl                    # Supervised Fine-Tuning (SFT) Trainer için

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 157.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170

## **İlk Denemeler**

In [ ]:
!pip install flash-attn --no-build-isolation

In [ ]:


import torch
from datasets import load_dataset ,Features, Value
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
import os
import gc

gc.collect()
torch.cuda.empty_cache()

# --- 1. AYARLAR ---
MODEL_NAME = "unsloth/llama-3-8b-bnb-4bit"
NEW_MODEL_NAME = "/content/drive/MyDrive/Eyüp Bağ - Tübitak 2209 - AP/models/medical_agent_llama3_v1(2000_Steps)"
DATA_PATH = "/content/drive/MyDrive/Eyüp Bağ - Tübitak 2209 - AP/Database/processed/combined_medical_train.jsonl"


def main():
    print(f"🚀 Eğitim Başlıyor (Final Fix)... Model: {MODEL_NAME}")

    # --- 2. 4-BIT QUANTIZATION ---
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    # --- 3. MODELİ YÜKLE ---
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )
    model.config.use_cache = False
    model.config.pretraining_tp = 1

    # --- 4. TOKENIZER ---
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # --- 5. LORA AYARLARI ---
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    )

    # --- 6. VERİ SETİ YÜKLEME ---
    my_features = Features({
        "instruction": Value("string"),
        "input": Value("string"),
        "output": Value("string")
    })

    print("Veri seti yükleniyor...")
    dataset = load_dataset(
        "json",
        data_files=DATA_PATH,
        split="train",
        features=my_features
    )
    def formatting_prompts_func(example):
        output_texts = []
        for instruction, input_text, output in zip(example['instruction'], example['input'], example['output']):
            instruction = str(instruction) if instruction is not None else ""
            input_text = str(input_text) if input_text is not None else ""
            output = str(output) if output is not None else ""

            if input_text.strip():
                text = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{output}<|end_of_text|>"
            else:
                text = f"### Instruction:\n{instruction}\n\n### Response:\n{output}<|end_of_text|>"
            output_texts.append(text)

        return {"text": output_texts}

    print("Veri formatlanıyor...")
    dataset = dataset.map(
        formatting_prompts_func,
        batched=True,
        remove_columns=dataset.column_names
    )

    print(f"İşlenmiş Veri Sayısı: {len(dataset)}")
    # --- 8. EĞİTİM ARGÜMANLARI (SFTConfig) ---
    training_arguments = SFTConfig(
        output_dir="./results",
        max_steps=2000,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=16,
        optim="paged_adamw_32bit",
        save_steps=200,
        logging_steps=50,
        learning_rate=5e-5,
        weight_decay=0.001,
        fp16=True,
        bf16=False,
        max_grad_norm=0.3,
        warmup_ratio=0.1,
        group_by_length=True,
        lr_scheduler_type="cosine",
        dataset_text_field="text",
        packing = False
    )

    # --- 9. TRAINER ---
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        peft_config=peft_config,
        processing_class=tokenizer,
        args=training_arguments,
    )

    print("🔥 Eğitim Başlıyor...")
    trainer.train()

    print(f"Model Kaydediliyor: {NEW_MODEL_NAME}")
    trainer.model.save_pretrained(NEW_MODEL_NAME)
    tokenizer.save_pretrained(NEW_MODEL_NAME)
    print("✅ Eğitim Tamamlandı!")

if __name__ == "__main__":
    main()

In [ ]:
import torch
from datasets import load_dataset, Features, Value
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig
import os
import gc

# --- BELLEK TEMİZLİĞİ ---
gc.collect()
torch.cuda.empty_cache()

# --- 1. AYARLAR ---
MODEL_NAME = "unsloth/llama-3-8b-bnb-4bit"
NEW_MODEL_NAME = "/content/drive/MyDrive/Eyüp Bağ - Tübitak 2209 - AP/models/medical_agent_llama3_v2_eval"
DATA_PATH = "/content/drive/MyDrive/Eyüp Bağ - Tübitak 2209 - AP/Database/processed/combined_medical_train.jsonl"

def main():
    print(f"🚀 Eğitim Başlıyor (Validation Loss + AMP + Cosine)...")

    # --- 2. 4-BIT QUANTIZATION ---
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    # --- 3. MODELİ YÜKLE ---
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )

    # Gradient Checkpointing (VRAM Tasarrufu)
    model.gradient_checkpointing_enable()
    model.config.use_cache = False
    model.config.pretraining_tp = 1

    model = prepare_model_for_kbit_training(model)

    # --- 4. TOKENIZER ---
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # --- 5. LORA AYARLARI ---
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    )

    model = get_peft_model(model, peft_config)

    # --- 6. VERİ SETİ YÜKLEME VE BÖLME (TRAIN/TEST SPLIT) ---
    print("Veri seti yükleniyor...")
    my_features = Features({
        "instruction": Value("string"),
        "input": Value("string"),
        "output": Value("string")
    })

    # Tüm veriyi önce 'train' split olarak yüklüyoruz
    full_dataset = load_dataset(
        "json",
        data_files=DATA_PATH,
        split="train",
        features=my_features
    )

    # --- YENİ EKLENEN KISIM: VERİYİ BÖLME ---
    print("Veri seti Eğitim (%95) ve Doğrulama (%5) olarak bölünüyor...")
    # 222.000 verinin %5'i (yaklaşık 11.000) test için ayrılacak.
    # seed=42 diyerek her seferinde aynı verilerin ayrılmasını sağlıyoruz.
    dataset_dict = full_dataset.train_test_split(test_size=0.05, seed=42)

    train_dataset = dataset_dict["train"]
    eval_dataset = dataset_dict["test"]

    print(f"Eğitim Verisi: {len(train_dataset)}")
    print(f"Doğrulama (Validation) Verisi: {len(eval_dataset)}")

    # --- 7. VERİ FORMATLAMA ---
    def formatting_prompts_func(example):
        output_texts = []
        for instruction, input_text, output in zip(example['instruction'], example['input'], example['output']):
            instruction = str(instruction) if instruction is not None else ""
            input_text = str(input_text) if input_text is not None else ""
            output = str(output) if output is not None else ""

            if input_text.strip():
                text = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{output}<|end_of_text|>"
            else:
                text = f"### Instruction:\n{instruction}\n\n### Response:\n{output}<|end_of_text|>"
            output_texts.append(text)

        return {"text": output_texts}

    print("Veri formatlanıyor...")
    # Hem train hem test setini aynı anda formatlar
    train_dataset = train_dataset.map(formatting_prompts_func, batched=True, remove_columns=train_dataset.column_names)
    eval_dataset = eval_dataset.map(formatting_prompts_func, batched=True, remove_columns=eval_dataset.column_names)

    # --- 8. EĞİTİM ARGÜMANLARI ---
    training_arguments = SFTConfig(
        output_dir="./results_v2_eval",

        # --- EĞİTİM STRATEJİSİ -f--
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        max_steps=600,

        # --- DOĞRULAMA (EVALUATION) AYARLARI ---
        eval_strategy="steps",       # Adım bazlı kontrol et
        eval_steps=50,               # Her 50 adımda bir sınava sok (Validation Loss hesapla)
        per_device_eval_batch_size=1, # Test ederken de 1'er 1'er bak
        do_eval=True,                # Değerlendirmeyi aktif et

        # --- OPTİMİZASYON ---
        fp16=True,
        bf16=False,
        optim="paged_adamw_32bit",
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,

        # --- DİĞER ---
        save_steps=100,
        logging_steps=10,
        weight_decay=0.001,
        max_grad_norm=0.3,
        group_by_length=True,
        dataset_text_field="text",
        packing=False,
        gradient_checkpointing=True,
    )

    # --- 9. TRAINER ---
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset, # <-- BURASI EKLENDİ (Sınav kağıtları)
        args=training_arguments,
        processing_class=tokenizer,
    )

    # Normalizasyon katmanlarını float32 yap
    for name, module in trainer.model.named_modules():
        if "norm" in name:
            module = module.to(torch.float32)

    print("🔥 Eğitim Başlıyor ")
    trainer.train()

    print(f"Model Kaydediliyor: {NEW_MODEL_NAME}")
    trainer.model.save_pretrained(NEW_MODEL_NAME)
    tokenizer.save_pretrained(NEW_MODEL_NAME)
    print("✅ Eğitim Tamamlandı!")

if __name__ == "__main__":
    main()

In [ ]:
import torch
from datasets import load_dataset, Features, Value
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig
import os
import gc

# --- BELLEK TEMİZLİĞİ ---
gc.collect()
torch.cuda.empty_cache()

# --- 1. AYARLAR ---
MODEL_NAME = "unsloth/llama-3-8b-bnb-4bit"
NEW_MODEL_NAME = "/content/drive/MyDrive/Eyüp Bağ - Tübitak 2209 - AP/models/medical_agent_llama3_A100_Final"
DATA_PATH = "/content/drive/MyDrive/Eyüp Bağ - Tübitak 2209 - AP/Database/processed/combined_medical_train.jsonl"

def main():
    print(f"🚀 A100 İle Final Eğitim Başlıyor (Flash Attn 2 + BF16 + Full Epoch)...")

    # --- 2. A100 İÇİN BF16 QUANTIZATION ---
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        # A100'ün ana dili bfloat16'dır. En yüksek performans buradadır.
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    # --- 3. MODELİ YÜKLE (FLASH ATTENTION İLE) ---
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        # A100'ün süper gücü: Flash Attention 2
        attn_implementation="flash_attention_2"
    )

    model.gradient_checkpointing_enable()
    model.config.use_cache = False
    model.config.pretraining_tp = 1

    model = prepare_model_for_kbit_training(model)

    # --- 4. TOKENIZER ---
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # --- 5. LORA AYARLARI ---
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    )

    model = get_peft_model(model, peft_config)

    # --- 6. VERİ SETİ ---
    print("Veri seti yükleniyor...")
    my_features = Features({
        "instruction": Value("string"),
        "input": Value("string"),
        "output": Value("string")
    })

    full_dataset = load_dataset(
        "json",
        data_files=DATA_PATH,
        split="train",
        features=my_features
    )

    # Test için 500 veri ayırıyoruz
    dataset_dict = full_dataset.train_test_split(test_size=500, seed=42)
    train_dataset = dataset_dict["train"]
    eval_dataset = dataset_dict["test"]

    # --- 7. FORMATLAMA ---
    def formatting_prompts_func(example):
        output_texts = []
        for instruction, input_text, output in zip(example['instruction'], example['input'], example['output']):
            instruction = str(instruction) if instruction is not None else ""
            input_text = str(input_text) if input_text is not None else ""
            output = str(output) if output is not None else ""

            if input_text.strip():
                text = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{output}<|end_of_text|>"
            else:
                text = f"### Instruction:\n{instruction}\n\n### Response:\n{output}<|end_of_text|>"
            output_texts.append(text)
        return {"text": output_texts}

    print("Veri formatlanıyor...")
    train_dataset = train_dataset.map(formatting_prompts_func, batched=True, remove_columns=train_dataset.column_names)
    eval_dataset = eval_dataset.map(formatting_prompts_func, batched=True, remove_columns=eval_dataset.column_names)

    # --- 8. EĞİTİM ARGÜMANLARI (A100 GÜCÜ) ---
    training_arguments = SFTConfig(
        output_dir="./results_A100_final",

        # --- STRATEJİ: FULL EPOCH ---
        # Artık "step" saymıyoruz. Verinin tamamını (1 Epoch) dönecek.
        num_train_epochs=1,

        # --- BATCH SIZE ---
        # A100 hafızası geniştir. Fiziksel olarak 8 veya 16 veri sığar.
        # Biz 8 seçelim, accumulation ile 64'e tamamlayalım. (Çok kararlı olur)
        per_device_train_batch_size=8,
        gradient_accumulation_steps=8, # 8x8 = 64 Effective Batch Size

        # --- HIZ VE KARARLILIK ---
        bf16=True,   # A100 için ZORUNLU (fp16 yerine)
        fp16=False,
        optim="paged_adamw_32bit",

        # Evaluation
        eval_strategy="steps",
        eval_steps=100, # Her 100 adımda bir kontrol
        per_device_eval_batch_size=8,
        do_eval=True,

        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,

        save_steps=200,
        logging_steps=25,
        weight_decay=0.001,
        max_grad_norm=0.3,
        group_by_length=True,
        max_length=1024,
        dataset_text_field="text",
        packing=False,
        gradient_checkpointing=True,
    )

    # --- 9. TRAINER ---
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=training_arguments,
        processing_class=tokenizer,
    )

    # Norm katmanlarını düzelt
    for name, module in trainer.model.named_modules():
        if "norm" in name:
            module = module.to(torch.float32)

    print("🔥 A100 ile Full Eğitim Başlıyor...")
    trainer.train()

    print(f"Model Kaydediliyor: {NEW_MODEL_NAME}")
    trainer.model.save_pretrained(NEW_MODEL_NAME)
    tokenizer.save_pretrained(NEW_MODEL_NAME)
    print("✅ Final Eğitim Tamamlandı!")

if __name__ == "__main__":
    main()

In [ ]:
# Temiz kurulum (önce çakışmaları sil)
!pip uninstall -y unsloth transformers trl peft accelerate bitsandbytes

# ZORUNLU ve STABİL sürümler
!pip install -U unsloth==2025.12.5
!pip install -U transformers==4.46.3
!pip install -U trl==0.11.4
!pip install -U peft==0.13.2
!pip install -U accelerate==0.34.2
!pip install -U datasets
!pip install -U bitsandbytes
!pip install -U tensorboard


In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes datasets


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-w_7ankbb/unsloth_0c508f603a31416497769ca5160bb11b
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-w_7ankbb/unsloth_0c508f603a31416497769ca5160bb11b
  Resolved https://github.com/unslothai/unsloth.git to commit 27185ebc4d9e430e28caf909144d64ffe3581a4a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.3/289.3 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.6/180.6 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 141.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 26.6 MB/s eta 0:0

In [ ]:
!pip uninstall -y unsloth transformers trl peft accelerate bitsandbytes
!pip install -U unsloth==2025.12.5
!pip install -U transformers==4.48.3
!pip install -U trl==0.11.4
!pip install -U peft==0.13.2
!pip install -U accelerate==0.34.2
!pip install -U datasets== 4.3.0
!pip install -U bitsandbytes
!pip install -U tensorboard


In [ ]:
# 1. Sorun çıkaran eski kütüphaneleri tamamen siliyoruz
!pip uninstall unsloth peft trl transformers accelerate -y

# 2. Unsloth'u en güncel haliyle yüklüyoruz
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 3. KRİTİK ADIM: Peft ve TRL'i en son sürüme ZORLUYORUZ
# SFTConfig kullanabilmek için trl'in güncel, peft'in de ona uyumlu olması şart.
!pip install --upgrade --no-cache-dir "trl>=0.9.0" "peft>=0.11.0" accelerate bitsandbytes

In [ ]:
!pip uninstall -y unsloth unsloth-zoo transformers trl peft accelerate bitsandbytes

!pip install -U unsloth==2025.12.5
!pip install -U transformers==4.48.3
!pip install -U trl==0.11.4
!pip install -U peft==0.17.0
!pip install -U accelerate==0.34.2
!pip install -U datasets==4.3.0 bitsandbytes tensorboard


In [ ]:
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()

# 1. Unsloth'u kur
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. Kritik kütüphaneleri SÜRÜM SABİTLEYEREK ve --no-deps ile kur
# Buradaki "trl<0.9.0" ifadesi hatayı çözen anahtardır.
if major_version >= 8:
    !pip install --no-deps packaging ninja einops flash-attn xformers "trl<0.9.0" peft accelerate bitsandbytes
else:
    !pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
import unsloth
from unsloth import FastLanguageModel

import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments
import os, logging, sys, gc


# --- BELLEK TEMİZLİĞİ ---
gc.collect()
torch.cuda.empty_cache()

# ==========================================
# ⚙️ 1. AYARLAR
# ==========================================
DATA_PATH = "/content/drive/MyDrive/Eyüp Bağ - Tübitak 2209 - AP/Database/processed/combined_medical_train.jsonl"
# Modeli kaydederken adım sayısını isme ekleyelim ki karışmasın
OUTPUT_DIR = "/content/drive/MyDrive/Eyüp Bağ - Tübitak 2209 - AP/models/medical_agent_llama3_2000steps"

MAX_SEQ_LENGTH = 2048
MAX_STEPS = 2000 # 🛑 EĞİTİMİ BURADA KESECEĞİZ

def main():

    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR, exist_ok=True)

    # --- LOGGING AYARLARI (KARA KUTU) ---
    log_file_path = os.path.join(OUTPUT_DIR, "training_log.txt")

    # Mevcut logger varsa temizle (Colab'de üst üste binmesin diye)
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    logging.basicConfig(
        format="%(asctime)s - %(levelname)s - %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
        level=logging.INFO,
        handlers=[
            logging.StreamHandler(sys.stdout), # Ekrana bas
            logging.FileHandler(log_file_path) # Dosyaya kaydet
        ]
    )
    logger = logging.getLogger(__name__)

    logger.info(f"✅ Log sistemi başlatıldı. Kayıt yeri: {log_file_path}")
    # ==========================================
    # 📥 2. MODEL VE TOKENIZER
    # ==========================================
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/llama-3-8b-bnb-4bit",
        max_seq_length = MAX_SEQ_LENGTH,
        dtype = None,
        load_in_4bit = True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 32,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )

    # ==========================================
    # 📝 3. FORMATLAMA (HİBRİT YAPI)
    # ==========================================

    EOS_TOKEN = tokenizer.eos_token

    # MEDİKAL AGENT ANAYASASI (System Prompt)
    # Bu metin her sorunun başına eklenecek.
    SYSTEM_PROMPT = """You are a highly knowledgeable and empathetic medical AI assistant named 'Maya'.
Your goal is to provide accurate, clinical, and evidence-based information.
Always maintain a professional tone.
If you are unsure about a diagnosis or if the information is critical, explicitly state uncertainty and recommend consulting a human doctor.
Do not make up medical facts."""

    def formatting_prompts_func(example):
        output_texts = []
        for i in range(len(example['instruction'])):
            instruction = example['instruction'][i]
            input_text = example['input'][i]
            output = example['output'][i]

            # 1. TEMEL YAPI: System + Instruction
            # "Input" varsa ekle, yoksa ekleme mantığı aynen devam ediyor.

            if input_text and str(input_text).strip() != "":
                # TEST SORUSU FORMATI (Input Var)
                text = f"### System:\n{SYSTEM_PROMPT}\n\n### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{output}{EOS_TOKEN}"
            else:
                # BİLGİ SORUSU FORMATI (Input Yok)
                text = f"### System:\n{SYSTEM_PROMPT}\n\n### Instruction:\n{instruction}\n\n### Response:\n{output}{EOS_TOKEN}"

            output_texts.append(text)
        return output_texts

    # ==========================================
    # 📂 4. VERİ YÜKLEME
    # ==========================================
    print("Veri seti yükleniyor...")
    dataset = load_dataset("json", data_files=DATA_PATH, split="train")

    # %1 Test ayır
    dataset = dataset.train_test_split(test_size=0.01, seed=42)
    train_dataset = dataset["train"]
    eval_dataset = dataset["test"]

    def compute_metrics(eval_preds):
        # Logits ve etiketler gelir
        logits, labels = eval_preds

        # Logits bir tuple ise (Unsloth bazen tuple döner) ilkini al
        if isinstance(logits, tuple):
            logits = logits[0]

        # Numpy -> Tensor dönüşümü (CPU'da hesapla ki GPU şişmesin)
        shift_logits = torch.tensor(logits[..., :-1, :]).float()
        shift_labels = torch.tensor(labels[..., 1:]).long()

        # Cross Entropy Loss hesapla
        loss_fct = torch.nn.CrossEntropyLoss(ignore_index=-100)
        loss = loss_fct(
            shift_logits.reshape(-1, shift_logits.size(-1)),
            shift_labels.reshape(-1)
        )

        # Perplexity = e^loss
        perplexity = torch.exp(loss)
        return {"perplexity": perplexity.item()}

    # ==========================================
    # 🔥 5. EĞİTİM ARGÜMANLARI (MAX_STEPS AYARLI)
    # ==========================================
    training_arguments = SFTConfig(
        output_dir = OUTPUT_DIR,
        per_device_train_batch_size = 16,
        per_device_eval_batch_size = 4,
        gradient_accumulation_steps = 2,  # Effective Batch = 32

        # --- DEĞİŞİKLİK BURADA ---
        max_steps = MAX_STEPS,   # Epoch yerine Adım sayısına göre duracak
        # num_train_epochs = 1,  # Bunu iptal ediyoruz, max_steps baskın gelir

        warmup_steps = 100,      # İlk 100 adım ısınma
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 50,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,

        report_to = ["tensorboard"], # "none" yerine bunu yazın
        logging_dir = os.path.join(OUTPUT_DIR, "runs"), # Grafikleri buraya kaydet

        # Sık sık değerlendirme yapalım ki 2000 adımın neresinde model iyileşti görelim
        eval_strategy="steps",
        eval_steps=250, # Her 250 adımda bir test et
        save_steps=250, # Her 500 adımda bir kaydet (Checkpoint)

        do_eval=True,
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        packing = False,
    )

    # ==========================================
    # 🏃 6. BAŞLAT
    # ==========================================
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = train_dataset,
        eval_dataset = eval_dataset,
        compute_metrics = compute_metrics,
        preprocess_logits_for_metrics = lambda logits, labels: logits,
        formatting_func = formatting_prompts_func,
        args = training_arguments,
    )

    print(f"🔥 UNLOTH İLE EĞİTİM BAŞLIYOR... (HEDEF: {MAX_STEPS} ADIM)")
    trainer.train()

    print(f"✅ Model Kaydediliyor: {OUTPUT_DIR}")
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print("🏁 TAMAMLANDI!")

if __name__ == "__main__":
    main()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


ImportError: cannot import name 'SFTConfig' from 'trl' (/usr/local/lib/python3.12/dist-packages/trl/__init__.py)

In [ ]:
!pip install \
  unsloth==2025.9.3 \
  transformers==4.57.3 \
  peft==0.17.0 \
  trl==0.11.4 \
  accelerate==0.34.2 \
  bitsandbytes==0.43.3 \
  triton==2.1.0


  Using cached unsloth-2025.9.3-py3-none-any.whl.metadata (52 kB)
  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
  Using cached peft-0.17.0-py3-none-any.whl.metadata (14 kB)
  Using cached trl-0.11.4-py3-none-any.whl.metadata (12 kB)
  Using cached accelerate-0.34.2-py3-none-any.whl.metadata (19 kB)
  Using cached bitsandbytes-0.43.3-py3-none-manylinux_2_24_x86_64.whl.metadata (3.5 kB)
ERROR: Ignored the following versions that require a different python version: 2025.3.4 Requires-Python <=3.12,>=3.9
ERROR: Could not find a version that satisfies the requirement triton==2.1.0 (from versions: 2.2.0, 2.3.0, 2.3.1, 3.0.0, 3.1.0, 3.2.0, 3.3.0, 3.3.1, 3.4.0, 3.5.0, 3.5.1)
ERROR: No matching distribution found for triton==2.1.0


In [ ]:
import bitsandbytes as bnb
print("bitsandbytes OK")

import unsloth
import transformers
print("Unsloth:", unsloth.__version__)
print("Transformers:", transformers.__version__)


ModuleNotFoundError: No module named 'bitsandbytes'

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
# SFTConfig SİLDİK, yerine TrainingArguments kullanıyoruz
from transformers import TrainingArguments
import os
import logging
import sys
import gc

# --- BELLEK TEMİZLİĞİ ---
gc.collect()
torch.cuda.empty_cache()

# ==========================================
# ⚙️ 1. AYARLAR
# ==========================================
DATA_PATH = "/content/drive/MyDrive/Eyüp Bağ - Tübitak 2209 - AP/Database/processed/combined_medical_train.jsonl"
OUTPUT_DIR = "/content/drive/MyDrive/Eyüp Bağ - Tübitak 2209 - AP/models/medical_agent_llama3_2000steps"
MAX_SEQ_LENGTH = 2048
MAX_STEPS = 2000

def main():
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR, exist_ok=True)

    # --- LOGGING AYARLARI ---
    log_file_path = os.path.join(OUTPUT_DIR, "training_log.txt")
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    logging.basicConfig(
        format="%(asctime)s - %(levelname)s - %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
        level=logging.INFO,
        handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(log_file_path)]
    )
    logger = logging.getLogger(__name__)
    logger.info(f"✅ Log sistemi başlatıldı. Kayıt yeri: {log_file_path}")

    # ==========================================
    # 📥 2. MODEL VE TOKENIZER
    # ==========================================
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/llama-3-8b-bnb-4bit",
        max_seq_length = MAX_SEQ_LENGTH,
        dtype = None,
        load_in_4bit = True,
        trust_remote_code = True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 32,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )

    # ==========================================
    # 📝 3. FORMATLAMA
    # ==========================================
    EOS_TOKEN = tokenizer.eos_token
    SYSTEM_PROMPT = """You are a highly knowledgeable and empathetic medical AI assistant named 'Maya'.
Your goal is to provide accurate, clinical, and evidence-based information.
Always maintain a professional tone.
If you are unsure about a diagnosis or if the information is critical, explicitly state uncertainty and recommend consulting a human doctor.
Do not make up medical facts."""

    def formatting_prompts_func(example):
        output_texts = []
        for i in range(len(example['instruction'])):
            instruction = example['instruction'][i]
            input_text = example['input'][i]
            output = example['output'][i]

            if input_text and str(input_text).strip() != "":
                text = f"### System:\n{SYSTEM_PROMPT}\n\n### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{output}{EOS_TOKEN}"
            else:
                text = f"### System:\n{SYSTEM_PROMPT}\n\n### Instruction:\n{instruction}\n\n### Response:\n{output}{EOS_TOKEN}"
            output_texts.append(text)
        return output_texts

    # ==========================================
    # 📂 4. VERİ YÜKLEME
    # ==========================================
    print("Veri seti yükleniyor...")
    dataset = load_dataset("json", data_files=DATA_PATH, split="train")
    dataset = dataset.train_test_split(test_size=0.01, seed=42)
    train_dataset = dataset["train"]
    eval_dataset = dataset["test"]

    # METRİKLER (Perplexity hesabı bazen OOM hatası verebilir, dikkatli olun)
    def compute_metrics(eval_preds):
        logits, labels = eval_preds
        if isinstance(logits, tuple): logits = logits[0]
        shift_logits = torch.tensor(logits[..., :-1, :]).float()
        shift_labels = torch.tensor(labels[..., 1:]).long()
        loss_fct = torch.nn.CrossEntropyLoss(ignore_index=-100)
        loss = loss_fct(shift_logits.reshape(-1, shift_logits.size(-1)), shift_labels.reshape(-1))
        return {"perplexity": torch.exp(loss).item()}

    # ==========================================
    # 🔥 5. EĞİTİM ARGÜMANLARI (REVİZE EDİLDİ)
    # ==========================================
    # SFTConfig yerine TrainingArguments kullanıyoruz!
    training_arguments = TrainingArguments(
        output_dir = OUTPUT_DIR,
        per_device_train_batch_size = 16,
        per_device_eval_batch_size = 4,
        gradient_accumulation_steps = 2,
        max_steps = MAX_STEPS,
        warmup_steps = 100,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 50,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        report_to = ["tensorboard"],
        logging_dir = os.path.join(OUTPUT_DIR, "runs"),
        eval_strategy="steps",
        eval_steps=250,
        save_steps=250,
        do_eval=True,
    )

    # ==========================================
    # 🏃 6. BAŞLAT
    # ==========================================
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = train_dataset,
        eval_dataset = eval_dataset,
        # compute_metrics = compute_metrics, # Hata alırsanız bunu yorum satırı yapın
        formatting_func = formatting_prompts_func,
        args = training_arguments,

        # SFTTrainer (Eski Sürüm) bu parametreleri burada ister:
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        packing = False,
    )

    print(f"🔥 UNSLOTH İLE EĞİTİM BAŞLIYOR... (HEDEF: {MAX_STEPS} ADIM)")
    trainer.train()

    print(f"✅ Model Kaydediliyor: {OUTPUT_DIR}")
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print("🏁 TAMAMLANDI!")

if __name__ == "__main__":
    main()

In [ ]:
# SADECE DEĞİŞKENLER HAFIZADAYSA ÇALIŞIR
try:
    SAVE_PATH = "/content/drive/MyDrive/Eyüp Bağ - Tübitak 2209 - AP/models/medical_agent_2000steps/checkpoint-250-rescue"
    model.save_pretrained(SAVE_PATH)
    tokenizer.save_pretrained(SAVE_PATH)
    print(f"✅ Mucize! Model kurtarıldı: {SAVE_PATH}")
except NameError:
    print("❌ Maalesef değişkenler silinmiş. Yeniden eğitim şart.")

In [ ]:
# --- BELLEK TEMİZLİĞİ ---
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset, Dataset, concatenate_datasets
from trl import SFTTrainer
from transformers import TrainingArguments
import pandas as pd
import json
import os
import re

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Model ve Tokenizer Yükleme
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# 2. PEFT (LoRA) Ayarları
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,1,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Unsloth optimizasyonu
    random_state = 42,
)

# 3. Veri Seti ve Formatlama (DÜZELTİLDİ)
dataset = load_dataset(
    "json",
    data_files = {"train": "/content/drive/MyDrive/Eyüp Bağ - Tübitak 2209 - AP/Database/processed/mayo_clinic/longform_train.jsonl"}
)["train"]

dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = dataset["train"]
eval_ds  = dataset["test"]

# EOS TOKEN EKLENDİ!
EOS_TOKEN = tokenizer.eos_token

def format_prompt(example):
    # Llama-3 standardına daha uygun basit format + EOS Token
    text = f"### Instruction:\n{example['prompt']}\n\n### Response:\n{example['response']}{EOS_TOKEN}"
    return {"text": text}

train_ds = train_ds.map(format_prompt)
eval_ds = eval_ds.map(format_prompt)

# 4. Eğitim Argümanları (HIZLANDIRILDI)
training_args = TrainingArguments(
    output_dir = "/content/drive/MyDrive/Eyüp Bağ - Tübitak 2209 - AP/models/maya_medical_lora",

    # --- PERFORMANS AYARLARI ---
    per_device_train_batch_size = 16, # 2 yerine 16 (A100 için). L4 ise 8 yapın.
    per_device_eval_batch_size = 8,
    gradient_accumulation_steps = 2,  # Effective Batch Size = 32 (16x2)
    num_train_epochs = 3,

    # --- OPTIMIZER ---
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    optim = "adamw_8bit", # Hafıza dostu optimizer
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    warmup_ratio = 0.05,

    # --- LOGGING & EVAL ---
    logging_steps = 10,
    eval_strategy = "steps",
    eval_steps = 100,
    save_steps = 100,
    report_to = "none",

    # En iyi modeli kaydetme
    load_best_model_at_end = True,
    metric_for_best_model = "eval_loss",
)

# 5. Trainer (METRIK HESAPLAMA SİLİNDİ)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = eval_ds,
    dataset_text_field = "text",
    max_seq_length = 2048,
    packing = False, # Packing bazen EOS token ile karışıklık yaratabilir, False daha güvenli.
    args = training_args,
    # compute_metrics SİLİNDİ -> Zaten trainer loss veriyor.
)

print("🚀 Eğitim Başlıyor...")
trainer.train()

In [ ]:
print(train_ds.column_names)


In [ ]:
def train_multi_lora():
    # 1. Verileri Hazırla
    preparer = MedicalDatasetPreparer()
    datasets_dict = preparer.prepare_all_datasets()

    # 2. Modeli Başlat
    model, tokenizer = initialize_model()

    # 3. Eğitim Argümanları (Genel)
    training_args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Gerçek eğitimde 1 epoch veya daha fazla step gerekir
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    )

    # 4. Döngüsel Eğitim (Sequential Adapter Training)
    for dataset_name, dataset in datasets_dict.items():
        if isinstance(dataset, list) and len(dataset) == 0: continue # Boş dataset kontrolü

        print(f"\n--- {dataset_name.upper()} için Eğitim Başlıyor ---\n")

        # Her dataset için özel çıktı dizini
        current_output_dir = f"outputs/{dataset_name}_lora"
        training_args.output_dir = current_output_dir

        trainer = SFTTrainer(
            model = model,
            processing_class = tokenizer,
            train_dataset = dataset,
            dataset_text_field = "text",
            max_length = 2048,
            dataset_num_proc = 2,
            packing = False,
            args = training_args,
        )

        trainer.train()

        # Adaptörü Kaydet
        print(f"Adaptör kaydediliyor: {current_output_dir}")
        model.save_pretrained(current_output_dir)
        tokenizer.save_pretrained(current_output_dir)

        # Not: Unsloth/PEFT mimarisinde, model objesi üzerindeki adaptör ağırlıkları güncellenir.
        # Bir sonraki dataset için temiz bir başlangıç yapmak istiyorsak,
        # adaptörü 'unload' etmeli veya modeli yeniden yüklemeliyiz.
        # Ancak sürekli öğrenme (Continuous Learning) isteniyorsa, mevcut ağırlıklar üzerine devam edilebilir.
        # Burada Multi-LoRA (ayrı adaptörler) istendiği için, ideal olan modeli resetlemektir.
        # Basitlik adına, bu kod bloğu ardışık ince ayarı (fine-tuning) gösterir.
        # Tam izolasyon için: model, tokenizer = initialize_model() her döngüde tekrar çağrılabilir.

if __name__ == "__main__":
    train_multi_lora()

# **Database updated**

In [ ]:
%%capture
# Unsloth'u en güncel ve kendi bağımlılıklarını yönetecek şekilde yüklüyoruz
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# Sadece datasets'i ayrı yüklüyoruz, diğerleri Unsloth tarafından yönetilmeli
!pip install datasets

In [ ]:
!pip install --upgrade unsloth trl peft accelerate xformers bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 49.2 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.3.0
    Uninstalling datasets-4.3.0:
      Successfully uninstalled datasets-4.3.0
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully uninstalled trl-0.24.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.2.1 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 4.6.1 which is incompatible.
unsloth-zoo 2026.2.1 requires trl!=0.19.0,<=0.24.0,>=0.18.2, but you have trl 0.29.0 which is incompatible.


In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import os
import logging
import sys
import gc

# ==========================================
# 🧹 0. BELLEK TEMİZLİĞİ (Colab Çökmelerini Önler)
# ==========================================
gc.collect()
torch.cuda.empty_cache()

# ==========================================
# ⚙️ 1. AYARLAR VE YOLLAR
# ==========================================
# 16.000 satırlık hazırladığın son JSONL dosyasının yolu
DATA_PATH = "/content/drive/MyDrive/Eyüp Bağ - Tübitak 2209 - AP/Database/processed/combined_medical_train_2.jsonl"
OUTPUT_DIR = "/content/drive/MyDrive/Eyüp Bağ - Tübitak 2209 - AP/models/maya_llama3_final"

# Makaledeki donanım kısıtlamalarına göre uyarlanmış bağlam uzunluğu
MAX_SEQ_LENGTH = 1024

def main():
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR, exist_ok=True)

    # --- LOGGING AYARLARI ---
    log_file_path = os.path.join(OUTPUT_DIR, "training_log.txt")
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    logging.basicConfig(
        format="%(asctime)s - %(levelname)s - %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
        level=logging.INFO,
        handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(log_file_path)]
    )
    logger = logging.getLogger(__name__)
    logger.info(f"✅ Log sistemi başlatıldı. Kayıt yeri: {log_file_path}")

    # ==========================================
    # 📥 2. MODEL VE TOKENIZER YÜKLEME
    # ==========================================
    logger.info("Model ve Tokenizer yükleniyor...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/llama-3-8b-bnb-4bit",
        max_seq_length = MAX_SEQ_LENGTH,
        dtype = None,
        load_in_4bit = True,
        trust_remote_code = True,
    )

    # --- LORA ADAPTASYONU (MAKALEYE GÖRE GÜNCELLENDİ) ---
    model = FastLanguageModel.get_peft_model(
        model,
        r = 8, # Makaledeki r değeri
        lora_alpha = 16, # Makaledeki alpha değeri
        target_modules = ["q_proj", "v_proj"], # Makaleye sadık kalarak sadece Query ve Value projection
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )

    # ==========================================
    # 📝 3. PROMPT FORMATLAMA (RAG Uyumlu Alpaca)
    # ==========================================
    EOS_TOKEN = tokenizer.eos_token
    # Maya'nın kimliği Sistem Promptuna sabitlendi
    SYSTEM_PROMPT = """You are a highly knowledgeable and empathetic medical AI assistant named 'Maya'.
Your goal is to provide accurate, clinical, and evidence-based information.
Always maintain a professional tone.
If you are unsure about a diagnosis or if the information is critical, explicitly state uncertainty and recommend consulting a human doctor.
Do not make up medical facts."""

    def formatting_prompts_func(example):
        output_texts = []
        for i in range(len(example['instruction'])):
            instruction = example['instruction'][i]
            input_text = example['input'][i]
            output = example['output'][i]

            if input_text and str(input_text).strip() != "":
                text = f"### System:\n{SYSTEM_PROMPT}\n\n### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{output}{EOS_TOKEN}"
            else:
                text = f"### System:\n{SYSTEM_PROMPT}\n\n### Instruction:\n{instruction}\n\n### Response:\n{output}{EOS_TOKEN}"
            output_texts.append(text)
        return output_texts

    # ==========================================
    # 📂 4. VERİ YÜKLEME VE BÖLME
    # ==========================================
    logger.info("Veri seti yükleniyor...")
    dataset = load_dataset("json", data_files=DATA_PATH, split="train")

    # Verinin %1'ini doğrulama (eval) için ayırıyoruz
    dataset = dataset.train_test_split(test_size=0.01, seed=42)
    train_dataset = dataset["train"]
    eval_dataset = dataset["test"]
    logger.info(f"Eğitim satır sayısı: {len(train_dataset)} | Test satır sayısı: {len(eval_dataset)}")

    # ==========================================
    # 🔥 5. EĞİTİM ARGÜMANLARI (MAKALEYE GÖRE GÜNCELLENDİ)
    # ==========================================
    training_arguments = TrainingArguments(
        output_dir = OUTPUT_DIR,
        num_train_epochs = 1,                 # 16.000 satır için 1 tam tur başlangıç için idealdir
        per_device_train_batch_size = 2,      # Makaledeki Batch Size
        gradient_accumulation_steps = 64,     # Makaledeki Gradient Accumulation
        learning_rate = 3e-4,                 # Makaledeki Learning Rate
        warmup_steps = 100,                   # Makaledeki Warmup adımı
        weight_decay = 0.01,                  # Makaledeki Weight Decay
        lr_scheduler_type = "cosine",         # Makaledeki Scheduler
        per_device_eval_batch_size = 2,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,                   # Her 10 adımda bir loss değerini gösterir
        optim = "adamw_8bit",
        seed = 3407,
        report_to = ["tensorboard"],
        logging_dir = os.path.join(OUTPUT_DIR, "runs"),
        eval_strategy = "steps",
        eval_steps = 50,                      # Her 50 adımda bir test verisinde başarıyı ölçer
        save_strategy = "epoch",              # Modeli adım bazlı değil, epoch bittiğinde kaydeder
        do_eval = True,
    )

    # ==========================================
    # 🏃 6. EĞİTİMİ BAŞLAT
    # ==========================================
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = train_dataset,
        eval_dataset = eval_dataset,
        formatting_func = formatting_prompts_func,
        args = training_arguments,
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        packing = False,
    )

    logger.info("🚀 UNSLOTH İLE EĞİTİM BAŞLIYOR...")
    trainer.train()

    # ==========================================
    # 💾 7. MODELİ KAYDET
    # ==========================================
    logger.info(f"✅ Eğitim bitti! Model kaydediliyor: {OUTPUT_DIR}")
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    logger.info("🏁 TÜM İŞLEMLER BAŞARIYLA TAMAMLANDI!")

if __name__ == "__main__":
    main()

2026-03-02 20:05:19 - INFO - ✅ Log sistemi başlatıldı. Kayıt yeri: /content/drive/MyDrive/Eyüp Bağ - Tübitak 2209 - AP/models/maya_llama3_final/training_log.txt
2026-03-02 20:05:19 - INFO - Model ve Tokenizer yükleniyor...
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch Attention layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch O projection layer with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.2.1 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


2026-03-02 20:06:18 - INFO - Veri seti yükleniyor...
2026-03-02 20:06:18 - INFO - Veri seti Llama-3 (Alpaca) formatına dönüştürülüyor...
2026-03-02 20:06:18 - INFO - Eğitim satır sayısı: 16165 | Test satır sayısı: 164
2026-03-02 20:06:19 - INFO - Unsloth: Padding-free batching auto-enabled for SFTTrainer instance.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/16165 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/164 [00:00<?, ? examples/s]

2026-03-02 20:06:58 - INFO - 🚀 UNSLOTH İLE EĞİTİM BAŞLIYOR...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 16,165 | Num Epochs = 1 | Total steps = 127
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 64
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 64 x 1) = 128
 "-____-"     Trainable parameters = 3,407,872 of 8,033,669,120 (0.04% trained)


OutOfMemoryError: CUDA out of memory. Tried to allocate 502.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 219.81 MiB is free. Including non-PyTorch memory, this process has 14.35 GiB memory in use. Of the allocated memory 14.17 GiB is allocated by PyTorch, and 18.79 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)